# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirl0w/Machine-Learning-Intern/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Before writing the rule, I checked two signals it depends on:

Signal 1 — Staleness (behind FlyRank's refresh flag): does days_since_last_update
relate to decline?
Signal 2 — CTR vs position (behind the CTR-fix flag): does CTR drop as position
tier worsens?

My rule: flag a page if it is (stale AND still visible) OR (visible AND has
weak CTR at a decent position).

Reason codes this rule can output:
- stale_visible_page
- low_ctr_visible_page

In [3]:
# Signal 1: staleness vs decline
df["stale_bucket"] = pd.cut(df["days_since_last_update"],
                              bins=[0,90,180,365,10000],
                              labels=["<90d","90-180d","180-365d","365d+"])
bucket1 = df.groupby("stale_bucket").agg(
    decline_rate=("trend_direction", lambda x: (x == "down").mean()),
    n=("trend_direction", "size"))
print("Signal 1 — Staleness vs decline:")
print(bucket1)

# Signal 2: CTR vs position tier
visible = df[df["impressions_90d"] >= 100]
bucket2 = visible.groupby("position_tier").agg(
    mean_ctr=("ctr", "mean"),
    n=("ctr", "size")).sort_values("mean_ctr", ascending=False)
print("\nSignal 2 — CTR vs position tier:")
print(bucket2)

Signal 1 — Staleness vs decline:
              decline_rate      n
stale_bucket                     
<90d              0.512031  20655
90-180d           0.611057   9171
180-365d          0.467456    169
365d+             0.600000      5

Signal 2 — CTR vs position tier:
               mean_ctr     n
position_tier                
page_1         0.354760  8633
top_3          0.334128   533
striking       0.255782  5903
page_3_5       0.142359  6058
deep           0.055415   879


/tmp/ipykernel_807/4192209847.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = df.groupby("stale_bucket").agg(


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Signal 1 verdict: [CONFIRMED/OPPOSITE/MIXED/FALSE] — [one sentence why, citing the numbers above]
Signal 2 verdict: [CONFIRMED/OPPOSITE/MIXED/FALSE] — [one sentence why, citing the numbers above]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
df["score"] = (
    ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int) * df["impressions_90d"] * 0.5
    + ((df["ctr"] < 0.3) & (df["impressions_90d"] >= 500)).astype(int) * df["impressions_90d"] * 0.5
)
df["reason_code"] = np.where(
    (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500),
    "stale_visible_page", "low_ctr_visible_page")
df["action"] = "review_for_refresh"

queue = df.sort_values("score", ascending=False)
os.makedirs("work/outputs", exist_ok=True)
queue[["content_id","score","reason_code","action"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)
print(f"Wrote {len(queue)} rows.")

Wrote 30000 rows.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top20 = queue.head(20)[["content_id","score","reason_code",
                          "impressions_90d","days_since_last_update","ctr","trend_direction"]]
top20

,content_id,score,reason_code,impressions_90d,days_since_last_update,ctr,trend_direction
6653,content_5fe46e04994d,258857.5,low_ctr_visible_page,517715,104,0.14,down
17812,content_aaef01a50def,258554.5,low_ctr_visible_page,517109,22,0.25,stable
26844,content_8c19996aa890,254626.0,low_ctr_visible_page,509252,20,0.15,down
19636,content_2cb567c3c89b,248863.5,low_ctr_visible_page,497727,48,0.10,up
29400,content_2dba2b1f9536,221717.0,low_ctr_visible_page,443434,104,0.21,stable
29879,content_1a9e894be2e2,208090.0,low_ctr_visible_page,416180,22,0.23,down
18870,content_db5989a78dd3,172555.5,low_ctr_visible_page,345111,20,0.21,up
26531,content_cb112fce36be,154955.0,low_ctr_visible_page,309910,104,0.16,down
3394,content_36ff89c8214e,147548.5,low_ctr_visible_page,295097,104,0.05,stable
26798,content_b28d1efd668f,143304.0,low_ctr_visible_page,286608,104,0.06,stable


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


1. content_id=XXXX — action: review_for_refresh, reason: stale_visible_page.
   Confidence: [high/medium]. Would be wrong if: [e.g. this page was recently
   consolidated with a sibling URL].
2. ...
(repeat through 20)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Weakest picks: rows #[X] and #[Y] — their scores are borderline, close to the
threshold, so they may be noise rather than real signal.

Leakage check: this rule uses only days_since_last_update, impressions_90d,
and ctr — all observable at decision time. No product flags (health_score,
priority_score) and no future-window data were used.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.